# 02 - Routine Scraping Notebook
## Instagram Competition Monitoring & Curation System

Notebook ini dijalankan secara berkala (rekomendasi: setiap Jumat pukul 12.00 WIB) untuk mengumpulkan postingan terbaru sejak `global_checkpoint` terakhir.
- Memeriksa keberadaan `global_checkpoint` (jika kosong, eksekusi dihentikan dengan arahan menjalankan Initial Scraping terlebih dahulu sesuai D13).
- Memprioritaskan Source Accounts, kemudian keyword aktif yang jatuh tempo berdasarkan kolom `Frequency`.
- Menyimpan raw post ke System Log sebelum AI extraction.
- Melakukan matching 4 atribut terhadap Competition Report tanpa merusak formula atau data admin manual.
- Memajukan `global_checkpoint` (waktu MULAI run) saat cakupan scraping lengkap (D18), walaupun eksekusi `Partial` karena kegagalan ekstraksi, asalkan minimal satu sumber selesai di-scrape; post yang gagal diambil ulang dari System Log tanpa scraping ulang.


### 0. Environment Check & Smart Install (wajib dijalankan pertama)
Sel pertama mencetak versi Python dan paket, lalu memeriksa import serta signature API yang dipakai `compmon`.
Sel kedua hanya memasang paket yang **belum ada atau terlalu lama** (tidak pernah `pip install -r`, tidak menurunkan paket bawaan Colab).
Jika ada paket yang dipasang: **Runtime > Restart session**, lalu jalankan ulang notebook dari atas.


In [ ]:
# @title 0a. Environment Check & Clone GitHub (jika di Colab)
# Memastikan repositori ter-clone di Colab, lalu memeriksa Python, versi paket, import, dan signature API.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/fajrialfarizy/SPIKE-Competition-Searching.git"
REPO_NAME = "SPIKE-Competition-Searching"

# Jika dijalankan di Google Colab (/content) dan compmon belum ada, clone dari GitHub
if not (Path.cwd() / 'compmon').exists() and not (Path.cwd().parent / 'compmon').exists():
    if not (Path.cwd() / REPO_NAME).exists():
        print(f"Mengunduh repositori dari GitHub: {REPO_URL}...")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    if (Path.cwd() / REPO_NAME).exists():
        os.chdir(Path.cwd() / REPO_NAME)
        print(f"Beralih ke direktori proyek: {os.getcwd()}")

_root = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from compmon.env_check import run_environment_check

run_environment_check()


In [ ]:
# @title 0b. Smart Install
# Hanya memasang paket yang belum ada / di bawah versi minimum di requirements.txt.
# Tidak memakai `pip install -r` dan tidak menurunkan paket bawaan Colab.
from compmon.env_check import ensure_environment

ensure_environment()


### 1. Setup Lingkungan & Autentikasi Google Sheets
Memuat kode modul `compmon` (dependensi sudah dicek dan dipasang oleh sel 0) dan melakukan autentikasi via `google.auth.default()` + `gspread.authorize()`.

In [ ]:
# @title 1. Muat Kode
import os, sys
from pathlib import Path

# Pastikan root project dan compmon masuk ke sys.path
root_dir = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from google.colab import auth, userdata
import google.auth
import gspread

# Autentikasi akun Google resmi Colab (Instruction 13)
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Membaca kredensial dari Colab Secrets
SPREADSHEET_ID = userdata.get('SPREADSHEET_ID')
APIFY_API_TOKEN = userdata.get('APIFY_API_TOKEN')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### 2. Eksekusi Routine Scraping & Pemrosesan AI

In [ ]:
from compmon.sheets_repo import SheetsRepository
from compmon.pipeline import CompetitionPipeline

repo = SheetsRepository(gc, SPREADSHEET_ID)
pipeline = CompetitionPipeline(
    sheets_repo=repo,
    apify_token=APIFY_API_TOKEN,
    gemini_api_key=GEMINI_API_KEY,
    notebook_name="ROUTINE"
)

summary = pipeline.run_routine_scraping()

print("=== Ringkasan Eksekusi Routine Scraping ===")
for key, val in summary.items():
    print(f"{key}: {val}")